# Fontes de dados — Enchentes em Porto Alegre

1. UFRGS/IPH - Banco de dados das cheias 2024 (Zenodo)
2. OpenStreetMap - escolas e unidades de saúde (Overpass)
3. ANA - nível do Guaíba (telemetria)

In [18]:
%pip install -q pandas numpy geopandas pyogrio shapely pyproj requests plotly folium matplotlib openpyxl nbformat unidecode

import os
import xml.etree.ElementTree as ET
import requests
import pandas as pd
import geopandas as gpd
import pyogrio
import plotly.express as px
import folium

os.makedirs("dados/originais", exist_ok=True)
HEADERS = {"User-Agent": "Fernando-Piedro-Victoria"}


[notice] A new release of pip is available: 26.2.1 -> 26.2.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## 1. UFRGS/IPH - Banco de dados das cheias 2024
https://doi.org/10.5281/zenodo.11164049 · licença CC-BY 4.0

In [ ]:
def baixar_zenodo(nome):
    arq = next(f for f in rec["files"] if f["key"] == nome)
    caminho = f"dados/originais/{nome}"
    if not os.path.exists(caminho):
        with open(caminho, "wb") as f:
            f.write(requests.get(arq["links"]["self"], headers=HEADERS, timeout=120).content)
    return caminho

arq_glossario = baixar_zenodo("cheias_rhguaiba_2024_db_v14_glossario.csv")
arq_abrigos = baixar_zenodo("rs_crise_abrigos_24052024_20h.csv")

### Glossário das camadas do banco de dados

In [21]:
glossario = pd.read_csv(arq_glossario)
glossario

,Camada,Categoria,Subcategoria,Abrangencia,Descricao,Fonte,Observacao,NotaTecnica
0,analise_ada_inundacao_20240506,Hidrologia,-,Região Hidrográfica do Lago Guaíba,Analise da area diretamente afetada (ADA) em 0...,Elaboracao Propria,NaN,NaN
1,poa_admin_bairros,Administracao,-,Porto Alegre,Bairros de Porto Alegre,Prefeitura Porto Alegre,-,-
2,poa_inundacao_1941_475cm,Hidrologia,Simulacao Hidroestatica,Porto Alegre,Mancha estimada de inudacao em Porto Alegre em...,Os autores,-,-
3,poa_inundacao_1967_313cm,Hidrologia,Simulacao Hidroestatica,Porto Alegre,Mancha estimada de inudacao em Porto Alegre em...,Os autores,-,-
4,poa_inundacao_2015_294cm,Hidrologia,Simulacao Hidroestatica,Porto Alegre,Mancha de inudacao simulada em Porto Alegre pa...,Os autores,-,-
5,poa_inundacao_2023_nov_346cm,Hidrologia,Simulacao Hidroestatica,Porto Alegre,Mancha estimada de inudacao em Porto Alegre em...,Os autores,-,-
6,poa_inundacao_2023_set_318cm,Hidrologia,Simulacao Hidroestatica,Porto Alegre,Mancha estimada de inudacao em Porto Alegre em...,Os autores,-,-
7,poa_inundacao_2024_530cm,Hidrologia,Simulacao Hidroestatica,Porto Alegre,Mancha de inudacao simulada em Porto Alegre pa...,Os autores,-,-
8,poa_inundacao_simulada_500cm,Hidrologia,Simulacao Hidroestatica,Porto Alegre,Mancha de inudacao simulada em Porto Alegre pa...,Os autores,-,-
9,poa_inundacao_simulada_550cm,Hidrologia,Simulacao Hidroestatica,Porto Alegre,Mancha de inudacao simulada em Porto Alegre pa...,Os autores,-,-


### Abrigos no RS em 24/05/2024, 20h

In [23]:
abrigos = pd.read_csv(arq_abrigos, sep=";")
abrigos.head()

,fid,NAME,TIPO DE LOCAL (VALIDADO UFRGS),TIPO DE LOCAL (NÃO VALIDADO),ENDEREÇO,O QUE ESTÁ FALTANDO,O QUE ESTÁ SOBRANDO,PRECISA DE VOLUNTÁRIOS,QUANTOS VOLUNTÁRIOS,PESSOAS ABRIGADAS,CAPACIDADE,ACEITA PETS,INFORMAÇÕES SOBRE PETS,ÚLTIMA ATUALIZAÇÃO,FONTE,CIDADE,CÓDIGO,LATITUDE,LONGITUDE
0,1,Associação Adote Viralata - Abrigo Pet,"Abrigo de animais, hotel ou hospital PET","Abrigo,Abrigo de Cachorros","R. Gen. Sampaio, 2 - Piratini, Alvorada - Rs, ...","Ração Gatos (alta),Ração Seca (alta),Casinhas ...",NaN,NaN,NaN,0,0,sim,NaN,5/24/2024 14:36,SOS RS,Alvorada,CD01130420,-30.011351,-51.042085
1,2,Olhos de quem ve,"Abrigo de animais, hotel ou hospital PET",NaN,17 de abril 150 Guajuviras canoas,NaN,NaN,NaN,NaN,NaN,NaN,sim,NaN,05/09/24,SOS RS,Canoas,CD89461346,-29.894686,-51.134627
2,3,American One/Gatos da Corte,"Abrigo de animais, hotel ou hospital PET",Abrigo de animais,"Av. Pedro Adams Filho, 4765 - Industrial, Novo...",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,05/11/24,Prefeitura de Novo Hamburgo,NaN,CD69061286,-29.690671,-51.128635
3,4,Hotel da FENAC,"Abrigo de animais, hotel ou hospital PET",Abrigo de animais,"R. Araxá, 755 - Ideal, Novo Hamburgo",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,15/05/24,Prefeitura de Novo Hamburgo,NaN,CD70371419,-29.703746,-51.141919
4,5,Fundação Escola Técnica Liberato Salzano Vieir...,"Abrigo de animais, hotel ou hospital PET",Abrigo de animais,"R. Inconfidentes, 395 - Primavera, Novo Hamburgo",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,24/05/24,Prefeitura de Novo Hamburgo,NaN,CD69231478,-29.692322,-51.147861


In [ ]:
px.scatter_map(abrigos, lat="LATITUDE", lon="LONGITUDE", color="TIPO DE LOCAL (VALIDADO UFRGS)",
               hover_name="NAME", hover_data=["CIDADE", "PESSOAS ABRIGADAS", "CAPACIDADE"],
               zoom=7, height=600, map_style="carto-positron", title="Abrigos — 24/05/2024")

## 2. OpenStreetMap — escolas e unidades de saúde
https://overpass-api.de · licença ODbL (© contribuidores do OpenStreetMap)

In [16]:
consulta = """
[out:json][timeout:120];
area["name"="Porto Alegre"]["admin_level"="8"]->.poa;
nwr["amenity"~"school|kindergarten|hospital|clinic|doctors"](area.poa);
out center tags;
"""
# o servidor principal costuma ficar sobrecarregado; tenta os espelhos em sequência
SERVIDORES = ["https://overpass-api.de/api/interpreter",
              "https://maps.mail.ru/osm/tools/overpass/api/interpreter",
              "https://overpass.private.coffee/api/interpreter"]

arq_osm = "dados/originais/osm_escolas_saude_poa.json"
if not os.path.exists(arq_osm):
    for url in SERVIDORES:
        try:
            r = requests.post(url, data={"data": consulta}, headers=HEADERS, timeout=300)
            r.json()   # falha se o servidor devolveu uma página de erro
            break
        except Exception as e:
            print("falhou:", url, "-", type(e).__name__)
    else:
        raise RuntimeError("nenhum servidor Overpass respondeu; tente de novo em alguns minutos")
    with open(arq_osm, "wb") as f:
        f.write(r.content)

osm = pd.json_normalize(pd.read_json(arq_osm, typ="series")["elements"])
osm.head(
)

,type,id,lat,lon,tags.addr:city,tags.addr:housenumber,tags.addr:postcode,tags.addr:street,tags.addr:suburb,tags.amenity,...,tags.reg_name,tags.addr:neighbourhood,tags.artist_name,tags.email_1,tags.education,tags.layer,tags.operator:short,tags.operator:wikidata,tags.source:addr,tags.type
0,node,476647429,-30.037137,-51.198245,Porto Alegre,400,90420-020,Rua Professor Álvaro Alvim,Rio Branco,hospital,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,node,598724565,-30.042889,-51.126231,NaN,NaN,NaN,NaN,NaN,clinic,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,node,1118102920,-30.130579,-51.239623,NaN,1959,91760-000,Coronel Marcos,NaN,school,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,node,1139635710,-30.034895,-51.180255,NaN,285,90470-340,Avenida Soledade,NaN,kindergarten,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,node,1139971906,-30.029053,-51.178558,NaN,1353,NaN,Avenida Doutor Nilo Peçanha,NaN,kindergarten,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [22]:
osm["lat"] = osm["lat"].fillna(osm["center.lat"])
osm["lon"] = osm["lon"].fillna(osm["center.lon"])
px.scatter_map(osm, lat="lat", lon="lon", color="tags.amenity", hover_name="tags.name",
               zoom=10, height=600, map_style="carto-positron")

## 3. ANA — nível do Guaíba (telemetria)
http://telemetriaws1.ana.gov.br · estação 87444000 (Usina do Gasômetro, cheia de 2024)

In [17]:
r = requests.get("http://telemetriaws1.ana.gov.br/ServiceANA.asmx/DadosHidrometeorologicos",
                 params={"codEstacao": 87444000, "dataInicio": "01/05/2024", "dataFim": "31/05/2024"},
                 headers=HEADERS, timeout=120)
root = ET.fromstring(r.content)
ana = pd.DataFrame([{c.tag: c.text for c in reg} for reg in root.iter("DadosHidrometereologicos")])
ana.head(50)

,CodEstacao,DataHora,Vazao,Nivel,Chuva
0,87444000,2024-05-31 23:45:00,None,365.00,0.00
1,87444000,2024-05-31 23:30:00,None,365.00,0.00
2,87444000,2024-05-31 23:15:00,None,365.00,0.00
3,87444000,2024-05-31 23:00:00,None,366.00,0.00
4,87444000,2024-05-31 22:45:00,None,366.00,0.00
5,87444000,2024-05-31 22:30:00,None,366.00,0.00
6,87444000,2024-05-31 22:15:00,None,366.00,0.00
7,87444000,2024-05-31 22:00:00,None,366.00,0.00
8,87444000,2024-05-31 21:45:00,None,366.00,0.00
9,87444000,2024-05-31 21:30:00,None,367.00,0.00


In [6]:
ana["DataHora"] = pd.to_datetime(ana["DataHora"])
ana["Nivel"] = pd.to_numeric(ana["Nivel"])
px.line(ana.sort_values("DataHora"), x="DataHora", y="Nivel", title="Nível do Guaíba (cm) — maio/2024")